In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

ratings = pd.read_csv("data/ratings.csv")
movies = pd.read_csv("data/movies.csv")

In [2]:
user_item_matrix = ratings.pivot(
    index="userId",
    columns="movieId",
    values="rating"
)

print("=== User-Item Matrix ===")
print(user_item_matrix.iloc[:5, :10])

print("\nMatrix shape:", user_item_matrix.shape)

=== User-Item Matrix ===
movieId   1   2    3   4   5    6   7   8   9   10
userId                                            
1        4.0 NaN  4.0 NaN NaN  4.0 NaN NaN NaN NaN
2        NaN NaN  NaN NaN NaN  NaN NaN NaN NaN NaN
3        NaN NaN  NaN NaN NaN  NaN NaN NaN NaN NaN
4        NaN NaN  NaN NaN NaN  NaN NaN NaN NaN NaN
5        4.0 NaN  NaN NaN NaN  NaN NaN NaN NaN NaN

Matrix shape: (610, 9724)


In [3]:
item_user_matrix = user_item_matrix.T

print("=== Item-User Matrix ===")
print(item_user_matrix.iloc[:10, :5])

print("\nMatrix shape:", item_user_matrix.shape)

=== Item-User Matrix ===
userId     1   2   3   4    5
movieId                      
1        4.0 NaN NaN NaN  4.0
2        NaN NaN NaN NaN  NaN
3        4.0 NaN NaN NaN  NaN
4        NaN NaN NaN NaN  NaN
5        NaN NaN NaN NaN  NaN
6        4.0 NaN NaN NaN  NaN
7        NaN NaN NaN NaN  NaN
8        NaN NaN NaN NaN  NaN
9        NaN NaN NaN NaN  NaN
10       NaN NaN NaN NaN  NaN

Matrix shape: (9724, 610)


In [4]:
movie_a_id = 1
movie_b_id = 2

movie_a = item_user_matrix.loc[movie_a_id]
movie_b = item_user_matrix.loc[movie_b_id]

print("=== Toy Story Ratings ===")
print(movie_a.dropna().head(10))

print("\n=== Jumanji Ratings ===")
print(movie_b.dropna().head(10))

=== Toy Story Ratings ===
userId
1     4.0
5     4.0
7     4.5
15    2.5
17    4.5
18    3.5
19    4.0
21    3.5
27    3.0
31    5.0
Name: 1, dtype: float64

=== Jumanji Ratings ===
userId
6     4.0
8     4.0
18    3.0
19    3.0
20    3.0
21    3.5
27    4.0
51    4.5
62    4.0
68    2.5
Name: 2, dtype: float64


In [5]:
common_ratings = pd.DataFrame({
    "Toy Story": movie_a,
    "Jumanji": movie_b
}).dropna()

print("=== Common User Ratings ===")
print(common_ratings.head(10))

print("\nNumber of common users:", len(common_ratings))

=== Common User Ratings ===
        Toy Story  Jumanji
userId                    
18            3.5      3.0
19            4.0      3.0
21            3.5      3.5
27            3.0      4.0
68            2.5      2.5
82            2.5      3.0
91            4.0      3.0
93            3.0      5.0
103           4.0      4.0
107           4.0      5.0

Number of common users: 68


In [6]:
ratings_a = common_ratings["Toy Story"].to_numpy()
ratings_b = common_ratings["Jumanji"].to_numpy()

similarity = cosine_similarity(
    ratings_a.reshape(1, -1),
    ratings_b.reshape(1, -1)
)[0][0]

print("=== Item-Based Cosine Similarity ===")
print("Movie A: Toy Story")
print("Movie B: Jumanji")
print("Common users:", len(common_ratings))
print("Cosine similarity:", similarity)

=== Item-Based Cosine Similarity ===
Movie A: Toy Story
Movie B: Jumanji
Common users: 68
Cosine similarity: 0.9644641106501716


In [7]:
pearson_similarity = common_ratings[
    "Toy Story"
].corr(
    common_ratings["Jumanji"]
)

print("=== Similarity Comparison ===")

print("Common users:", len(common_ratings))
print("Cosine similarity:", similarity)
print("Pearson correlation:", pearson_similarity)

=== Similarity Comparison ===
Common users: 68
Cosine similarity: 0.9644641106501716
Pearson correlation: 0.33097835454242513


In [11]:
def find_similar_movies(
    movie_id,
    item_user_matrix,
    movies,
    min_common_users=10,
    top_n=10,
    shrinkage=20
):
    target_ratings = item_user_matrix.loc[movie_id]

    results = []

    for other_movie_id in item_user_matrix.index:

        if other_movie_id == movie_id:
            continue

        other_ratings = item_user_matrix.loc[other_movie_id]

        common = pd.DataFrame({
            "target": target_ratings,
            "other": other_ratings
        }).dropna()

        n_common = len(common)

        if n_common < min_common_users:
            continue

        similarity = common["target"].corr(
            common["other"]
        )

        if pd.isna(similarity):
            continue

        adjusted_similarity = (
            n_common / (n_common + shrinkage)
        ) * similarity

        results.append({
            "movieId": other_movie_id,
            "similarity": similarity,
            "common_users": n_common,
            "adjusted_similarity": adjusted_similarity
        })

    results = pd.DataFrame(
        results,
        columns=[
            "movieId",
            "similarity",
            "common_users",
            "adjusted_similarity"
        ]
    )

    results = results.merge(
        movies[["movieId", "title"]],
        on="movieId",
        how="left"
    )

    results = results.sort_values(
        by=["adjusted_similarity", "common_users"],
        ascending=[False, False]
    )

    return results.head(top_n)

In [12]:
similar_movies_adjusted = find_similar_movies(
    movie_id=1,
    item_user_matrix=item_user_matrix,
    movies=movies,
    min_common_users=10,
    top_n=10,
    shrinkage=20
)

print("=== Adjusted Similar Movies ===")

print(
    similar_movies_adjusted[
        [
            "title",
            "similarity",
            "common_users",
            "adjusted_similarity"
        ]
    ].to_string(index=False)
)

=== Adjusted Similar Movies ===
                                      title  similarity  common_users  adjusted_similarity
                         Toy Story 2 (1999)    0.699211            81             0.560753
                             Aladdin (1992)    0.611892           107             0.515531
                    Incredibles, The (2004)    0.643301            77             0.510662
                        Finding Nemo (2003)    0.618701            87             0.503056
                   Santa Clause, The (1994)    0.569712            47             0.399649
                      Monsters, Inc. (2001)    0.490231            88             0.399448
                     Erin Brockovich (2000)    0.598016            40             0.398677
                         Toy Story 3 (2010)    0.577446            44             0.396994
Wallace & Gromit: The Wrong Trousers (1993)    0.589625            40             0.393083
                       Arachnophobia (1990)    0.652424   

In [13]:
similar_movies_20 = find_similar_movies(
    movie_id=1,
    item_user_matrix=item_user_matrix,
    movies=movies,
    min_common_users=20,
    top_n=10,
    shrinkage=20
)

print("=== Similar Movies: Minimum 20 Common Users ===")

print(
    similar_movies_20[
        [
            "title",
            "similarity",
            "common_users",
            "adjusted_similarity"
        ]
    ].to_string(index=False)
)

=== Similar Movies: Minimum 20 Common Users ===
                                      title  similarity  common_users  adjusted_similarity
                         Toy Story 2 (1999)    0.699211            81             0.560753
                             Aladdin (1992)    0.611892           107             0.515531
                    Incredibles, The (2004)    0.643301            77             0.510662
                        Finding Nemo (2003)    0.618701            87             0.503056
                   Santa Clause, The (1994)    0.569712            47             0.399649
                      Monsters, Inc. (2001)    0.490231            88             0.399448
                     Erin Brockovich (2000)    0.598016            40             0.398677
                         Toy Story 3 (2010)    0.577446            44             0.396994
Wallace & Gromit: The Wrong Trousers (1993)    0.589625            40             0.393083
                       Arachnophobia (1990

In [14]:
rating_history = {
    2571: 5.0,  # The Matrix
    1: 4.5,     # Toy Story
    2: 2.0      # Jumanji
}

baseline = 3.5

print("=== User Rating History ===")
print(rating_history)

=== User Rating History ===
{2571: 5.0, 1: 4.5, 2: 2.0}


In [15]:
def calculate_item_similarity(
    movie_a_id,
    movie_b_id,
    item_user_matrix,
    min_common_users=10,
    shrinkage=20
):
    if movie_a_id not in item_user_matrix.index:
        return np.nan

    if movie_b_id not in item_user_matrix.index:
        return np.nan

    movie_a = item_user_matrix.loc[movie_a_id]
    movie_b = item_user_matrix.loc[movie_b_id]

    common = pd.DataFrame({
        "movie_a": movie_a,
        "movie_b": movie_b
    }).dropna()

    n_common = len(common)

    if n_common < min_common_users:
        return np.nan

    similarity = common["movie_a"].corr(
        common["movie_b"]
    )

    if pd.isna(similarity):
        return np.nan

    adjusted_similarity = (
        n_common / (n_common + shrinkage)
    ) * similarity

    return adjusted_similarity

In [16]:
similarity = calculate_item_similarity(
    movie_a_id=1,
    movie_b_id=2,
    item_user_matrix=item_user_matrix,
    min_common_users=10,
    shrinkage=20
)

print("=== Toy Story vs Jumanji ===")
print("Adjusted similarity:", similarity)

=== Toy Story vs Jumanji ===
Adjusted similarity: 0.2557560012373285


In [17]:
def calculate_cf_score(
    candidate_movie_id,
    rating_history,
    item_user_matrix,
    baseline=3.5,
    min_common_users=10,
    shrinkage=20
):
    numerator = 0.0
    denominator = 0.0

    for rated_movie_id, rating in rating_history.items():

        similarity = calculate_item_similarity(
            movie_a_id=candidate_movie_id,
            movie_b_id=rated_movie_id,
            item_user_matrix=item_user_matrix,
            min_common_users=min_common_users,
            shrinkage=shrinkage
        )

        if pd.isna(similarity):
            continue

        preference = rating - baseline

        numerator += similarity * preference
        denominator += abs(similarity)

    if denominator == 0:
        return np.nan

    return numerator / denominator

In [18]:
candidate_movie_id = 3114

cf_score = calculate_cf_score(
    candidate_movie_id=candidate_movie_id,
    rating_history=rating_history,
    item_user_matrix=item_user_matrix,
    baseline=3.5,
    min_common_users=10,
    shrinkage=20
)

print("=== Collaborative Filtering Score ===")
print("Movie: Toy Story 2")
print("CF Score:", cf_score)

=== Collaborative Filtering Score ===
Movie: Toy Story 2
CF Score: 0.7146200153128466


In [24]:
def calculate_all_item_similarities(
    movie_id,
    user_item_matrix,
    min_common_users=10,
    shrinkage=20
):
    # 1. Get ratings for the target movie
    target_ratings = user_item_matrix[movie_id]

    # 2. Identify users who rated the target movie
    target_users = target_ratings.dropna().index

    # 3. Get all movie ratings from those users
    subset = user_item_matrix.loc[target_users]

    # 4. Count common raters
    common_counts = subset.notna().sum(axis=0)

    # 5. Keep movies with sufficient common raters
    valid_movies = common_counts[
        common_counts >= min_common_users
    ].index

    subset_valid = subset[valid_movies]

    # 6. Check rating variance among common raters for both movies
    target_common = pd.DataFrame(
        {
            col: target_ratings.loc[target_users]
            for col in valid_movies
        }
    ).where(subset_valid.notna())

    valid_variance = (
        (subset_valid.std(axis=0) > 0)
        & (target_common.std(axis=0) > 0)
    )

    subset_valid = subset_valid.loc[
        :, valid_variance
    ]

    # 7. Calculate Pearson correlations
    similarities = subset_valid.corrwith(
        target_ratings.loc[target_users],
        axis=0
    )

    # 8. Apply shrinkage
    adjusted = (
        similarities
        * common_counts.loc[similarities.index]
        / (
            common_counts.loc[similarities.index]
            + shrinkage
        )
    )

    # 9. Restore the full movie index
    results = pd.Series(
        np.nan,
        index=user_item_matrix.columns,
        dtype=float
    )

    results.loc[adjusted.index] = adjusted

    # 10. Exclude the target movie itself
    results.loc[movie_id] = np.nan

    return results

In [20]:
all_similarities = calculate_all_item_similarities(
    movie_id=1,
    user_item_matrix=user_item_matrix,
    min_common_users=10,
    shrinkage=20
)

print("=== Similarity Validation ===")

print(
    "Toy Story vs Jumanji:",
    all_similarities.loc[2]
)

print(
    "Toy Story vs Toy Story 2:",
    all_similarities.loc[3114]
)

print(
    "Number of valid similarities:",
    all_similarities.notna().sum()
)

c:\Users\Hanbd\miniconda3\envs\graph_table\Lib\site-packages\numpy\lib\_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
c:\Users\Hanbd\miniconda3\envs\graph_table\Lib\site-packages\numpy\lib\_function_base_impl.py:2888: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
c:\Users\Hanbd\miniconda3\envs\graph_table\Lib\site-packages\numpy\lib\_function_base_impl.py:2888: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
c:\Users\Hanbd\miniconda3\envs\graph_table\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\Hanbd\miniconda3\envs\graph_table\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


=== Similarity Validation ===
Toy Story vs Jumanji: 0.2557560012373285
Toy Story vs Toy Story 2: 0.5607533793072035
Number of valid similarities: 1691


In [21]:
def recommend_cf_movies(
    rating_history,
    user_item_matrix,
    movies,
    baseline=3.5,
    min_common_users=10,
    shrinkage=20,
    top_n=10
):
    # 1. Get all movie IDs
    movie_ids = user_item_matrix.columns

    # 2. Initialize numerator and denominator
    numerator = pd.Series(0.0, index=movie_ids)
    denominator = pd.Series(0.0, index=movie_ids)

    # 3. Calculate scores for each movie rated by the user
    for rated_movie_id, rating in rating_history.items():

        similarities = calculate_all_item_similarities(
            movie_id=rated_movie_id,
            user_item_matrix=user_item_matrix,
            min_common_users=min_common_users,
            shrinkage=shrinkage
        )

        preference = rating - baseline

        # Exclude undefined similarities from the sum
        valid_similarities = similarities.fillna(0.0)

        numerator += valid_similarities * preference
        denominator += valid_similarities.abs()

    # 4. Calculate CF recommendation scores
    cf_scores = numerator / denominator.replace(0, np.nan)

    # 5. Create the result DataFrame
    results = pd.DataFrame({
        "movieId": cf_scores.index,
        "cf_score": cf_scores.values
    })

    # 6. Join movie titles
    results = results.merge(
        movies[["movieId", "title", "genres"]],
        on="movieId",
        how="left"
    )

    # 7. Exclude already-rated movies
    results = results[
        ~results["movieId"].isin(rating_history.keys())
    ]

    # 8. Exclude movies with unavailable scores
    results = results.dropna(subset=["cf_score"])

    # 9. Sort by CF score in descending order
    results = results.sort_values(
        by="cf_score",
        ascending=False
    )

    return results.head(top_n)

In [22]:
rating_history = {
    2571: 5.0,
    1: 4.5,
    2: 2.0
}

cf_recommendations = recommend_cf_movies(
    rating_history=rating_history,
    user_item_matrix=user_item_matrix,
    movies=movies,
    baseline=3.5,
    min_common_users=10,
    shrinkage=20,
    top_n=10
)

print("=== Collaborative Filtering Top 10 ===")

print(
    cf_recommendations[
        [
            "title",
            "genres",
            "cf_score"
        ]
    ].to_string(index=False)
)

c:\Users\Hanbd\miniconda3\envs\graph_table\Lib\site-packages\numpy\lib\_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
c:\Users\Hanbd\miniconda3\envs\graph_table\Lib\site-packages\numpy\lib\_function_base_impl.py:2888: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
c:\Users\Hanbd\miniconda3\envs\graph_table\Lib\site-packages\numpy\lib\_function_base_impl.py:2888: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
c:\Users\Hanbd\miniconda3\envs\graph_table\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\Hanbd\miniconda3\envs\graph_table\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\Hanbd\miniconda3\envs\graph_table\Lib\site-packages\numpy\lib\_function_base_impl.py:

=== Collaborative Filtering Top 10 ===
                           title                        genres  cf_score
Adventures of Tintin, The (2011) Action|Animation|Mystery|IMAX       1.5
                Rambo III (1988) Action|Adventure|Thriller|War       1.5
            Shallow Grave (1994)         Comedy|Drama|Thriller       1.5
           Hurricane, The (1999)                         Drama       1.5
                   Jaws 2 (1978)               Horror|Thriller       1.5
      Sydney (Hard Eight) (1996)          Crime|Drama|Thriller       1.5
      Save the Last Dance (2001)                 Drama|Romance       1.5
            Touch of Evil (1958)      Crime|Film-Noir|Thriller       1.5
             Body of Lies (2008)         Action|Drama|Thriller       1.5
                Mist, The (2007)                 Horror|Sci-Fi       1.5


In [23]:
cf_all = recommend_cf_movies(
    rating_history=rating_history,
    user_item_matrix=user_item_matrix,
    movies=movies,
    baseline=3.5,
    min_common_users=10,
    shrinkage=20,
    top_n=len(user_item_matrix.columns)
)

toy_story_2 = cf_all[
    cf_all["movieId"] == 3114
]

print("=== Toy Story 2 CF Validation ===")
print(toy_story_2[["title", "cf_score"]])

print(
    "\nNumber of movies with CF scores:",
    len(cf_all)
)

print(
    "Minimum CF score:",
    cf_all["cf_score"].min()
)

print(
    "Maximum CF score:",
    cf_all["cf_score"].max()
)

c:\Users\Hanbd\miniconda3\envs\graph_table\Lib\site-packages\numpy\lib\_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
c:\Users\Hanbd\miniconda3\envs\graph_table\Lib\site-packages\numpy\lib\_function_base_impl.py:2888: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
c:\Users\Hanbd\miniconda3\envs\graph_table\Lib\site-packages\numpy\lib\_function_base_impl.py:2888: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
c:\Users\Hanbd\miniconda3\envs\graph_table\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\Hanbd\miniconda3\envs\graph_table\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\Hanbd\miniconda3\envs\graph_table\Lib\site-packages\numpy\lib\_function_base_impl.py:

=== Toy Story 2 CF Validation ===
                   title  cf_score
2353  Toy Story 2 (1999)   0.71462

Number of movies with CF scores: 1943
Minimum CF score: -1.5000000000000002
Maximum CF score: 1.5000000000000002


In [25]:
similarity_data = {}

for rated_movie_id in rating_history:

    similarity_data[rated_movie_id] = (
        calculate_all_item_similarities(
            movie_id=rated_movie_id,
            user_item_matrix=user_item_matrix,
            min_common_users=10,
            shrinkage=20
        )
    )

similarity_df = pd.DataFrame(similarity_data)

# Count valid similarities by movie
support_count = similarity_df.notna().sum(axis=1)

# Sum absolute similarities by movie
similarity_strength = similarity_df.abs().sum(axis=1)

print("=== CF Support Statistics ===")
print(support_count.value_counts().sort_index())

=== CF Support Statistics ===
0    7774
1     299
2     386
3    1265
Name: count, dtype: int64


In [26]:
cf_analysis = cf_all.copy()

cf_analysis["support_count"] = (
    cf_analysis["movieId"]
    .map(support_count)
)

cf_analysis["similarity_strength"] = (
    cf_analysis["movieId"]
    .map(similarity_strength)
)

print("=== CF Recommendation Diagnostics ===")

print(
    cf_analysis[
        [
            "title",
            "cf_score",
            "support_count",
            "similarity_strength"
        ]
    ].head(10).to_string(index=False)
)

=== CF Recommendation Diagnostics ===
                           title  cf_score  support_count  similarity_strength
Adventures of Tintin, The (2011)       1.5              1             0.224440
                Rambo III (1988)       1.5              1             0.187277
            Shallow Grave (1994)       1.5              1             0.053347
           Hurricane, The (1999)       1.5              1             0.097726
                   Jaws 2 (1978)       1.5              1             0.113576
      Sydney (Hard Eight) (1996)       1.5              1             0.120246
      Save the Last Dance (2001)       1.5              1             0.113488
            Touch of Evil (1958)       1.5              1             0.053670
             Body of Lies (2008)       1.5              1             0.198849
                Mist, The (2007)       1.5              1             0.048030


In [27]:
cf_filtered = cf_analysis[
    cf_analysis["support_count"] >= 2
].copy()

cf_filtered = cf_filtered.sort_values(
    by=[
        "cf_score",
        "similarity_strength"
    ],
    ascending=[False, False]
)

print("=== Improved CF Top 10 ===")

print(
    cf_filtered[
        [
            "title",
            "cf_score",
            "support_count",
            "similarity_strength"
        ]
    ].head(10).to_string(index=False)
)

=== Improved CF Top 10 ===
                                    title  cf_score  support_count  similarity_strength
Mission: Impossible - Rogue Nation (2015)  1.500000              2             0.233290
                        Hellraiser (1987)  1.500000              2             0.206512
                    Johnny English (2003)  1.500000              2             0.194660
                        Inside Man (2006)  1.494266              3             0.094741
                     Punisher, The (2004)  1.488047              2             0.041986
                 Snakes on a Plane (2006)  1.486738              3             0.252460
                       The Martian (2015)  1.481316              3             0.209708
                          Red Dawn (1984)  1.480927              2             0.282120
                          Bad Boys (1995)  1.480731              3             0.146820
         The Count of Monte Cristo (2002)  1.480339              3             0.273132


In [28]:
# 1. Use only positive similarities
positive_similarities = similarity_df.clip(lower=0).fillna(0)

# 2. Calculate preference deviations from user ratings
preferences = pd.Series({
    movie_id: rating - 3.5
    for movie_id, rating in rating_history.items()
})

# 3. Calculate the weighted sum
numerator = positive_similarities.mul(
    preferences, axis=1
).sum(axis=1)

# 4. Sum similarities
denominator = positive_similarities.sum(axis=1)

# 5. Apply confidence adjustment
lambda_cf = 0.5

improved_cf_scores = (
    numerator / (denominator + lambda_cf)
)

# 6. Count the positive similarities actually used
positive_support = (
    positive_similarities > 0
).sum(axis=1)

print("=== Improved CF Score Validation ===")
print("Minimum score:", improved_cf_scores.min())
print("Maximum score:", improved_cf_scores.max())
print("Movies with positive similarity:", (positive_support > 0).sum())

=== Improved CF Score Validation ===
Minimum score: -0.5441514223022103
Maximum score: 0.6613679662392545
Movies with positive similarity: 1786


In [29]:
cf_results = pd.DataFrame({
    "movieId": improved_cf_scores.index,
    "cf_score": improved_cf_scores.values,
    "positive_support": positive_support.values,
    "similarity_strength": denominator.values
})

# Join movie metadata
cf_results = cf_results.merge(
    movies[["movieId", "title", "genres"]],
    on="movieId",
    how="left"
)

# Exclude already-rated movies
cf_results = cf_results[
    ~cf_results["movieId"].isin(rating_history.keys())
]

# Keep movies supported by at least two positive similarities
cf_results = cf_results[
    cf_results["positive_support"] >= 2
]

# Sort by CF score in descending order
cf_results = cf_results.sort_values(
    by=["cf_score", "similarity_strength"],
    ascending=[False, False]
)

print("=== Stabilized CF Top 10 ===")

print(
    cf_results[
        [
            "title",
            "cf_score",
            "positive_support",
            "similarity_strength"
        ]
    ].head(10).to_string(index=False)
)

=== Stabilized CF Top 10 ===
                                    title  cf_score  positive_support  similarity_strength
                 Finding Neverland (2004)  0.661368                 2             0.418766
                           Memento (2000)  0.649032                 3             0.625929
                   St. Elmo's Fire (1985)  0.648182                 2             0.513331
                    Ice Storm, The (1997)  0.630508                 2             0.502098
        Untitled Spider-Man Reboot (2017)  0.629719                 2             0.533861
                             Shaft (2000)  0.628892                 2             0.398203
                    Apartment, The (1960)  0.624565                 2             0.474796
Batman v Superman: Dawn of Justice (2016)  0.621165                 2             0.451391
                     28 Days Later (2002)  0.620061                 3             0.474614
                           Ed Wood (1994)  0.613766          